<a href="https://colab.research.google.com/github/nt189/Regresion-Lineal-Simple-y-Multiple-/blob/main/Regresi%C3%B3n_lineal_presentacion_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Carga de librerias y dataframe

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from turtle import color
from sklearn.linear_model import LinearRegression

In [2]:
pais = 'mexico'
# pais = 'la_california'
df = pd.read_csv('listings_' + pais + '.csv')

In [3]:
# Elimino las columnas que no voy a utilizar
colums = ['room_type',
'price',
'host_acceptance_rate',
'host_is_superhost',
'accommodates',
'bathrooms',
'review_scores_cleanliness',
'host_identity_verified',
'instant_bookable',
'property_type']

df = df[colums]

df.head(3)

,room_type,price,host_acceptance_rate,host_is_superhost,accommodates,bathrooms,review_scores_cleanliness,host_identity_verified,instant_bookable,property_type
0,Entire home/apt,"$4,044.00",NaN,f,2,1.0,NaN,t,f,Entire villa
1,Entire home/apt,"$18,000.00",92%,t,14,5.5,4.70,t,f,Entire home
2,Entire home/apt,"$2,123.00",56%,t,4,1.0,4.76,t,f,Entire rental unit


# Tratamiento de nulo

In [4]:
# Vemos que columnas tienen valores nulos
df.isnull().sum()

,0
room_type,0
price,3815
host_acceptance_rate,2773
host_is_superhost,1382
accommodates,0
bathrooms,3804
review_scores_cleanliness,3297
host_identity_verified,3
instant_bookable,0
property_type,0


In [5]:
df['price'] = df['price'].replace('[\$,]', '', regex=True).astype(float)
df['host_acceptance_rate'] = df['host_acceptance_rate'].replace('%', '', regex=True).astype(float)

df['host_acceptance_rate'] = df['host_acceptance_rate'].fillna(round(df['host_acceptance_rate'].median(), 1))
df['price'] = df['price'].fillna(round(df['price'].median(), 1))
df['host_is_superhost'] = df['host_is_superhost'].fillna('f')
df['bathrooms'] = df['bathrooms'].fillna(method = 'ffill')
df['host_identity_verified'] = df['host_identity_verified'].fillna(method = "bfill")
df['review_scores_cleanliness'] = df['review_scores_cleanliness'].fillna(round(df['review_scores_cleanliness'].mean(),1))

<ipython-input-5-3d1c15f981cc>:7: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['bathrooms'] = df['bathrooms'].fillna(method = 'ffill')
<ipython-input-5-3d1c15f981cc>:8: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['host_identity_verified'] = df['host_identity_verified'].fillna(method = "bfill")


In [6]:
df.isnull().sum()

,0
room_type,0
price,0
host_acceptance_rate,0
host_is_superhost,0
accommodates,0
bathrooms,0
review_scores_cleanliness,0
host_identity_verified,0
instant_bookable,0
property_type,0


In [7]:
df['host_acceptance_rate']

,host_acceptance_rate
0,99.0
1,92.0
2,56.0
3,94.0
4,99.0
...,...
26276,100.0
26277,100.0
26278,92.0
26279,100.0


# Convercion de variables categoricas a numericas

In [8]:
# Obtenemos infomacion del dataframe
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26281 entries, 0 to 26280
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   room_type                  26281 non-null  object 
 1   price                      26281 non-null  float64
 2   host_acceptance_rate       26281 non-null  float64
 3   host_is_superhost          26281 non-null  object 
 4   accommodates               26281 non-null  int64  
 5   bathrooms                  26281 non-null  float64
 6   review_scores_cleanliness  26281 non-null  float64
 7   host_identity_verified     26281 non-null  object 
 8   instant_bookable           26281 non-null  object 
 9   property_type              26281 non-null  object 
dtypes: float64(4), int64(1), object(5)
memory usage: 2.0+ MB


In [9]:
room_type_mapping = dict(enumerate(df["room_type"].astype("category").cat.categories))
print(room_type_mapping)
df["room_type"] = df["room_type"].astype("category").cat.codes

{0: 'Entire home/apt', 1: 'Hotel room', 2: 'Private room', 3: 'Shared room'}


In [10]:
categorias = {"f": 0, "t": 1}
df["host_is_superhost"] = df["host_is_superhost"].map(categorias)
df["host_identity_verified"] = df["host_identity_verified"].map(categorias)
df["instant_bookable"] = df["instant_bookable"].map(categorias)

In [11]:
property_type_mapping = dict(enumerate(df["property_type"].astype("category").cat.categories))
print(property_type_mapping)
df["property_type"] = df["property_type"].astype("category").cat.codes

{0: 'Boat', 1: 'Campsite', 2: 'Casa particular', 3: 'Castle', 4: 'Dome', 5: 'Earthen home', 6: 'Entire bungalow', 7: 'Entire cabin', 8: 'Entire chalet', 9: 'Entire condo', 10: 'Entire cottage', 11: 'Entire guest suite', 12: 'Entire guesthouse', 13: 'Entire home', 14: 'Entire home/apt', 15: 'Entire hostel', 16: 'Entire in-law', 17: 'Entire loft', 18: 'Entire place', 19: 'Entire rental unit', 20: 'Entire serviced apartment', 21: 'Entire townhouse', 22: 'Entire vacation home', 23: 'Entire villa', 24: 'Farm stay', 25: 'Holiday park', 26: 'Hut', 27: 'Private room', 28: 'Private room in barn', 29: 'Private room in bed and breakfast', 30: 'Private room in cabin', 31: 'Private room in casa particular', 32: 'Private room in castle', 33: 'Private room in chalet', 34: 'Private room in condo', 35: 'Private room in cottage', 36: 'Private room in dome', 37: 'Private room in earthen home', 38: 'Private room in farm stay', 39: 'Private room in floor', 40: 'Private room in guest suite', 41: 'Private ro

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26281 entries, 0 to 26280
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   room_type                  26281 non-null  int8   
 1   price                      26281 non-null  float64
 2   host_acceptance_rate       26281 non-null  float64
 3   host_is_superhost          26281 non-null  int64  
 4   accommodates               26281 non-null  int64  
 5   bathrooms                  26281 non-null  float64
 6   review_scores_cleanliness  26281 non-null  float64
 7   host_identity_verified     26281 non-null  int64  
 8   instant_bookable           26281 non-null  int64  
 9   property_type              26281 non-null  int8   
dtypes: float64(4), int64(4), int8(2)
memory usage: 1.7 MB


# Tratamiento para valores atipicos

In [ ]:
# Creo 2 dataframe para poder procesar los outliers
cuantitativas = df.select_dtypes(include=[np.number])
cualitativas = df.select_dtypes(include=['object', 'category'])

In [ ]:
df['host_acceptance_rate'].unique()

array([ 99.,  92.,  56.,  94.,  97.,  75.,  96.,  82., 100.,  98.,  80.,
        58.,   0.,  83.,  25.,  79.,  40.,  29.,  50.,  95.,  70.,  63.,
        89.,  67.,  61.,  91.,  90.,  64.,  59.,  65.,  42.,  93.,  87.,
        19.,  33.,  74.,  69.,  23.,  14.,  60.,  62.,  78.,  86.,  85.,
        76.,  73.,  43.,  39.,  88.,  57.,  11.,  44.,  71.,  54.,  68.,
         9.,  17.,  36.,  81.,  52.,  15.,  10.,  51.,  72.,  84.,   8.,
        66.,  31.,  20.,  30.,  41.,  53.,  47.,  38.,   5.,  48.,  32.,
        77.,  27.,  35.,  13.,  46.,  37.,  21.,  22.,   2.,  18.,   4.,
        55.,   7.,  24.,  28.,  12.,  49.,   1.,   6.,   3.,  45.,  26.])

In [ ]:
# Metodo aplicando cuantiles. Encuentro cuartiles 0.25 y 0.75
y = cuantitativas

percentiles25  = y.quantile(0.25)
percentiles75 = y.quantile(0.75)
iqr = percentiles75 - percentiles25

Limite_Superior_iqr = percentiles75 + 1.5*iqr
Limite_Inferior_iqr = percentiles25 - 1.5*iqr

print("Limite superior permitido")
print(Limite_Superior_iqr)
print("Limite inferior permitido")
print(Limite_Inferior_iqr)

Limite superior permitido
room_type                       5.000
price                        3126.500
host_acceptance_rate          106.000
host_is_superhost               2.500
accommodates                    7.000
bathrooms                       3.500
review_scores_cleanliness       5.255
host_identity_verified          1.000
instant_bookable                2.500
property_type                  76.500
dtype: float64
Limite inferior permitido
room_type                     -3.000
price                       -677.500
host_acceptance_rate          90.000
host_is_superhost             -1.500
accommodates                  -1.000
bathrooms                     -0.500
review_scores_cleanliness      4.415
host_identity_verified         1.000
instant_bookable              -1.500
property_type                -15.500
dtype: float64


In [ ]:
# Obtenemos datos limpios del DataFrame
Datos_sin_Outliers_iqr = cuantitativas[(y <= Limite_Superior_iqr) & (y >= Limite_Inferior_iqr)]

In [ ]:
# Remplazamos valores atipicos del dataframe con mean
df_clean_iqr = Datos_sin_Outliers_iqr.copy()
df_clean_iqr = df_clean_iqr.fillna(round(Datos_sin_Outliers_iqr.mean(),1))

In [ ]:
# contar nulos por columnas
df_clean_iqr.isnull().sum()

,0
room_type,0
price,0
host_acceptance_rate,0
host_is_superhost,0
accommodates,0
bathrooms,0
review_scores_cleanliness,0
host_identity_verified,0
instant_bookable,0
property_type,0


In [ ]:
df_clean_iqr['host_acceptance_rate'].unique()

array([ 99. ,  92. ,  98.8,  94. ,  97. ,  96. , 100. ,  98. ,  95. ,
        91. ,  90. ,  93. ])

In [ ]:
df = df_clean_iqr.copy()

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26281 entries, 0 to 26280
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   room_type                  26281 non-null  int8   
 1   price                      26281 non-null  float64
 2   host_acceptance_rate       26281 non-null  float64
 3   host_is_superhost          26281 non-null  int64  
 4   accommodates               26281 non-null  float64
 5   bathrooms                  26281 non-null  float64
 6   review_scores_cleanliness  26281 non-null  float64
 7   host_identity_verified     26281 non-null  float64
 8   instant_bookable           26281 non-null  int64  
 9   property_type              26281 non-null  float64
dtypes: float64(7), int64(2), int8(1)
memory usage: 1.8 MB


# Separación de tipos de habitaciones

In [13]:
room_type_mapping

{0: 'Entire home/apt', 1: 'Hotel room', 2: 'Private room', 3: 'Shared room'}

In [14]:
entire = df.copy()
private = df.copy()
shared = df.copy()
hotel = df.copy()

In [15]:
for key, value in room_type_mapping.items():
  if value == 'Entire home/apt':
    entire = entire[entire['room_type'] == key]
  elif value == 'Private room':
    private = private[private['room_type'] == key]
  elif value == 'Shared room':
    shared = shared[shared['room_type'] == key]
  elif value == 'Hotel room':
    hotel = hotel[hotel['room_type'] == key]

In [16]:
# entire.to_csv(pais + '_entire.csv')
# private.to_csv(pais + '_private.csv')
# shared.to_csv(pais + '_shared.csv')
# hotel.to_csv(pais + '_hotel.csv')

# Regrecion lineal

In [35]:
def reglineal(var_dep, vars_indep, dfa, retrn_tblpred = True):
  model = LinearRegression()
  model.fit(X = vars_indep, y = var_dep)
  print('Modelo matematico: y = ' + str(model.coef_) + ' | X = ' + str(model.intercept_))
  print('Coeficiencia del modelo apartir del coeficiente R (coeficiente de determinacion): ' + str(model.score(vars_indep, var_dep)))
  print('Coeficiente de correlacion ' + str(np.sqrt(model.score(vars_indep, var_dep))))
  y_pred = model.predict(X = vars_indep)

  if retrn_tblpred:
    dfx = dfa.copy()
    dfx.insert(0, 'predicciones', y_pred)
    return dfx
  else:
    return y_pred



# Graficos

In [36]:
def grafico(var_dep, vars_indep, dataf):
  sns.scatterplot(x=vars_indep, y = var_dep, color = "blue", data = dataf) #X varibles independientes, Y variable dependiente
  sns.scatterplot(x=vars_indep, y = 'predicciones', color = "red", data = dataf)
  plt.title(var_dep + ' vs ' + vars_indep)
  plt.show()

  # sns.pairplot(dataf)
  # plt.show()

  dataf = dataf.drop(['room_type'], axis = 1)
  dataf = abs(dataf.corr())
  plt.figure(figsize=(10, 8))
  sns.heatmap(dataf, cmap = "Grays", annot = True, fmt = ".4f")
  plt.show()

# Todos los graficos

In [ ]:
grafico('price', 'host_acceptance_rate', reglineal(entire['price'], entire[['host_acceptance_rate']], entire))
grafico('price', 'host_is_superhost', reglineal(entire['price'], entire[['host_is_superhost']], entire))
grafico('accommodates', 'bathrooms', reglineal(entire['accommodates'], entire[['bathrooms']], entire))
grafico('price', 'review_scores_cleanliness', reglineal(entire['price'], entire[['review_scores_cleanliness']], entire))
grafico('price', 'host_identity_verified', reglineal(entire['price'], entire[['host_identity_verified']], entire))
grafico('price', 'instant_bookable', reglineal(entire['price'], entire[['instant_bookable']], entire))
grafico('price', 'property_type', reglineal(entire['price'], entire[['property_type']], entire))

In [ ]:
grafico('price', 'host_acceptance_rate', reglineal(private['price'], private[['host_acceptance_rate']], private))
# grafico('price', 'host_is_superhost', reglineal(private['price'], private[['host_is_superhost']], private))
# grafico('accommodates', 'bathrooms', reglineal(private['accommodates'], private[['bathrooms']], private))
# grafico('price', 'review_scores_cleanliness', reglineal(private['price'], private[['review_scores_cleanliness']], private))
# grafico('price', 'host_identity_verified', reglineal(private['price'], private[['host_identity_verified']], private))
# grafico('price', 'instant_bookable', reglineal(private['price'], private[['instant_bookable']], private))
# grafico('price', 'property_type', reglineal(private['price'], private[['property_type']], private))

In [ ]:
grafico('price', 'host_acceptance_rate', reglineal(shared['price'], shared[['host_acceptance_rate']], shared))
# grafico('price', 'host_is_superhost', reglineal(shared['price'], shared[['host_is_superhost']], shared))
# grafico('accommodates', 'bathrooms', reglineal(shared['accommodates'], shared[['bathrooms']], shared))
# grafico('price', 'review_scores_cleanliness', reglineal(shared['price'], shared[['review_scores_cleanliness']], shared))
# grafico('price', 'host_identity_verified', reglineal(shared['price'], shared[['host_identity_verified']], shared))
# grafico('price', 'instant_bookable', reglineal(shared['price'], shared[['instant_bookable']], shared))
# grafico('price', 'property_type', reglineal(shared['price'], shared[['property_type']], shared))

In [ ]:
grafico('price', 'host_acceptance_rate', reglineal(hotel['price'], hotel[['host_acceptance_rate']], hotel))
# grafico('price', 'host_is_superhost', reglineal(hotel['price'], hotel[['host_is_superhost']], hotel))
# grafico('accommodates', 'bathrooms', reglineal(hotel['accommodates'], hotel[['bathrooms']], hotel))
# grafico('price', 'review_scores_cleanliness', reglineal(hotel['price'], hotel[['review_scores_cleanliness']], hotel))
# grafico('price', 'host_identity_verified', reglineal(hotel['price'], hotel[['host_identity_verified']], hotel))
# grafico('price', 'instant_bookable', reglineal(hotel['price'], hotel[['instant_bookable']], hotel))
# grafico('price', 'property_type', reglineal(hotel['price'], hotel[['property_type']], hotel))

# Un solo grafico

In [ ]:
entire_pred = entire.copy()
private_pred = private.copy()
shared_pred = shared.copy()
hotel_pred = hotel.copy()

In [ ]:
entire_pred = reglineal(entire_pred['price'], entire_pred[['host_acceptance_rate']], entire_pred, retrn_tblpred = False)

# Regrecion lineal multiple